In [3]:
from openpyxl import load_workbook


def write_to_excel(book, plot_name, product, value, sheet_name):
    sheet = book[sheet_name]

    # 找到对应的行
    row = None
    for idx, row_data in enumerate(sheet.iter_rows(min_row=2, values_only=True)):
        if row_data[1] == plot_name:
            row = idx + 2  # 加2是因为第一行是表头
            break
    if row is None:
        return

    # 找到对应的列
    col = None
    for idx, cell in enumerate(sheet[1]):
        if cell.value == product:
            col = idx  # 列索引
            break
    if col is None:
        return

    # 写入数据
    sheet.cell(row=row, column=col + 1, value=value)  # column+1是因为第一列是季度

In [4]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum

# 读取附件中的数据
attachment_1 = pd.read_excel('附件1.xlsx')  # 包含地块信息
attachment_2 = pd.read_excel('附件2.xlsx')  # 包含作物信息和 2023 年的统计数据

plots = attachment_1[['地块名称', '地块类型', '地块面积/亩']].to_dict('records')
crops = attachment_2[['作物名称', '作物类型', '种植面积/亩']].to_dict('records')

# 定义优化问题
model = LpProblem(name="Crop-Optimization", sense=LpMaximize)

years = range(2024, 2031)  # 从 2024 年到 2030 年

# 决策变量：x[i][j][t] --> 第 t 年地块 i 种植作物 j 的面积
x = LpVariable.dicts("x",
                     ((i['地块名称'], j['作物名称'], t) for i in plots for j in crops for t in years),
                     lowBound=0,
                     cat="Continuous")

# 辅助变量：z[i][j][t] --> 实际销售的作物产量（不能超过总产量和预期销售量的最小值）
z = LpVariable.dicts("z",
                     ((i['地块名称'], j['作物名称'], t) for i in plots for j in crops for t in years),
                     lowBound=0,
                     cat="Continuous")

# 二进制变量：y[i][j][t] --> 第 t 年地块 i 是否种植作物 j（1 表示种植，0 表示不种植）
y = LpVariable.dicts("y",
                     ((i['地块名称'], j['作物名称'], t) for i in plots for j in crops for t in years),
                     lowBound=0,
                     upBound=1, cat="Binary")

# 目标函数：最大化收益
model += lpSum(z[i['地块名称'], j['作物名称'], t] * j['种植面积/亩'] - x[i['地块名称'], j['作物名称'], t] * j['种植面积/亩'] for i in plots for j in crops for t in years)

# 约束条件
# 1. 每个地块每年的总种植面积不能超过其实际面积
for i in plots:
    for t in years:
        model += lpSum(x[i['地块名称'], j['作物名称'], t] for j in crops) <= i['地块面积/亩']

# 2. 作物实际销售产量 z[i][j][t] 不能超过作物的总产量和预期销售量
for i in plots:
    for j in crops:
        for t in years:
            model += z[i['地块名称'], j['作物名称'], t] <= x[i['地块名称'], j['作物名称'], t] * j['种植面积/亩']
            model += z[i['地块名称'], j['作物名称'], t] <= j['种植面积/亩']

# 3. 不重茬约束：同一地块不能连续两年种植相同作物
# 使用二进制变量 y 来表示是否种植
for i in plots:
    for j in crops:
        for t in range(2025, 2031):  # 确保 t 和 t-1 之间没有种植相同作物
            model += y[i['地块名称'], j['作物名称'], t] + y[i['地块名称'], j['作物名称'], t - 1] <= 1

# 4. 每三年内必须种植一次豆类作物
for i in plots:
    for t in range(2024, 2028):
        model += lpSum(y[i['地块名称'], j['作物名称'], t + k] for j in crops if j['作物类型'] == '豆类' for k in range(3)) >= 1

# 5. 面积约束：如果某地块某年种植某作物，则种植面积必须大于 0
for i in plots:
    for j in crops:
        for t in years:
            model += x[i['地块名称'], j['作物名称'], t] <= y[i['地块名称'], j['作物名称'], t] * i['地块面积/亩']

# 求解模型
status = model.solve()


book = load_workbook("result1_1(4).xlsx")

for i in plots:
    for j in crops:
        for t in years:
            if x[i['地块名称'], j['作物名称'], t].value() > 0:
                # write_to_excel(book, i['地块名称'], j['作物名称'], x[i['地块名称'], j['作物名称'], t].value(), str(t))
                print(f"地块 {i['地块名称']} 在第 {t} 年种植 {j['作物名称']} {x[i['地块名称'], j['作物名称'], t].value()} 亩")
                
book.save("result1_1(4).xlsx")

C:\Dev\DevKit\anaconda3\envs\machine-learning\lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


地块 A1 在第 2024 年种植 小麦 1.0 亩
地块 A1 在第 2025 年种植 小麦 1.0 亩
地块 A1 在第 2026 年种植 小麦 1.0 亩
地块 A1 在第 2027 年种植 小麦 1.0 亩
地块 A1 在第 2028 年种植 小麦 1.0 亩
地块 A1 在第 2029 年种植 小麦 1.0 亩
地块 A1 在第 2030 年种植 小麦 1.0 亩
地块 A1 在第 2024 年种植 玉米 1.0 亩
地块 A1 在第 2025 年种植 玉米 1.0 亩
地块 A1 在第 2026 年种植 玉米 1.0 亩
地块 A1 在第 2027 年种植 玉米 1.0 亩
地块 A1 在第 2028 年种植 玉米 1.0 亩
地块 A1 在第 2029 年种植 玉米 1.0 亩
地块 A1 在第 2030 年种植 玉米 1.0 亩
地块 A1 在第 2024 年种植 玉米 1.0 亩
地块 A1 在第 2025 年种植 玉米 1.0 亩
地块 A1 在第 2026 年种植 玉米 1.0 亩
地块 A1 在第 2027 年种植 玉米 1.0 亩
地块 A1 在第 2028 年种植 玉米 1.0 亩
地块 A1 在第 2029 年种植 玉米 1.0 亩
地块 A1 在第 2030 年种植 玉米 1.0 亩
地块 A1 在第 2024 年种植 黄豆 1.0 亩
地块 A1 在第 2025 年种植 黄豆 1.0 亩
地块 A1 在第 2026 年种植 黄豆 1.0 亩
地块 A1 在第 2027 年种植 黄豆 1.0 亩
地块 A1 在第 2028 年种植 黄豆 1.0 亩
地块 A1 在第 2029 年种植 黄豆 1.0 亩
地块 A1 在第 2030 年种植 黄豆 1.0 亩
地块 A1 在第 2024 年种植 绿豆 1.0 亩
地块 A1 在第 2025 年种植 绿豆 1.0 亩
地块 A1 在第 2026 年种植 绿豆 1.0 亩
地块 A1 在第 2027 年种植 绿豆 1.0 亩
地块 A1 在第 2028 年种植 绿豆 1.0 亩
地块 A1 在第 2029 年种植 绿豆 1.0 亩
地块 A1 在第 2030 年种植 绿豆 1.0 亩
地块 A1 在第 2024 年种植 谷子 1.0 亩
地块 A1 在第 2025 年种植 谷子 1.0 亩
地